In [1]:
import digitalhub as dh

project = dh.get_or_create_project("translation-tutorial")

In [2]:
project.log_artifact("train-data", kind="artifact", source="training.tar.gz")

/home/raman/python3.12/lib/python3.12/site-packages/digitalhub/entities/artifact/crud.py:134: UserWarning: log_artifact in version 0.16 wil log an artifact of kind 'artifact'.
  warn("log_artifact in version 0.16 wil log an artifact of kind 'artifact'.")
/home/raman/python3.12/lib/python3.12/site-packages/digitalhub/entities/artifact/artifact/crud.py:61: UserWarning: This method will become log_artifact in 0.16
  warn("This method will become log_artifact in 0.16")


{'kind': 'artifact', 'metadata': {'project': 'translation-tutorial', 'name': 'train-data', 'version': 'solid-penguin', 'created': '2026-07-01T14:59:01.779Z', 'updated': '2026-07-01T14:59:03.049Z', 'created_by': 'raman@fbk.eu', 'updated_by': 'raman@fbk.eu', 'embedded': False}, 'spec': {'path': 's3://dev-datalake/translation-tutorial/artifact/train-data/f18355a5f20348fca8b1faecc32854b0/training.tar.gz'}, 'status': {'state': 'READY', 'files': []}, 'user': 'raman@fbk.eu', 'project': 'translation-tutorial', 'name': 'train-data', 'id': 'f18355a5f20348fca8b1faecc32854b0', 'key': 'store://translation-tutorial/artifact/artifact/train-data:f18355a5f20348fca8b1faecc32854b0', 'extensions': []}

In [3]:
project.log_artifact("validation-data", kind="artifact", source="validation.tar.gz")

{'kind': 'artifact', 'metadata': {'project': 'translation-tutorial', 'name': 'validation-data', 'version': 'ultra-mole', 'created': '2026-07-01T14:59:44.755Z', 'updated': '2026-07-01T14:59:45.001Z', 'created_by': 'raman@fbk.eu', 'updated_by': 'raman@fbk.eu', 'embedded': False}, 'spec': {'path': 's3://dev-datalake/translation-tutorial/artifact/validation-data/cedcac1659d740e59766f715c5e42206/validation.tar.gz'}, 'status': {'state': 'READY', 'files': []}, 'user': 'raman@fbk.eu', 'project': 'translation-tutorial', 'name': 'validation-data', 'id': 'cedcac1659d740e59766f715c5e42206', 'key': 'store://translation-tutorial/artifact/artifact/validation-data:cedcac1659d740e59766f715c5e42206', 'extensions': []}

In [23]:
func = project.new_function(
    "train-translator",
    kind="python",
    python_version="PYTHON3_12",
    code_src="git+https://github.com/scc-digitalhub/digitalhub-tutorials",
    handler="torch-translation-tutorial.wrapper:train",
    requirements=["torch==2.3.0", "torchtext==0.18.0", "torchdata==0.9.0", "spacy", "portalocker", "click >= 8.2.1"]
)

In [27]:
run = func.run(
    action="job",
    inputs={
        "training_data": project.get_artifact("train-data").key,
        "validation_data": project.get_artifact("validation-data").key
    },
    parameters={
        "epochs": 1,
        "backend": "gpu",
        "src_lang": "de_core_news_sm"
    },
    local_execution=True
)

2026-07-01 16:19:50,245 - dhcore./home/raman/python3.12/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Validating task.
2026-07-01 16:19:50,246 - dhcore./home/raman/python3.12/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Starting task.
2026-07-01 16:19:50,247 - dhcore./home/raman/python3.12/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Configuring execution.
2026-07-01 16:19:51,596 - dhcore./home/raman/python3.12/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Composing function arguments.
2026-07-01 16:19:51,621 - dhcore./home/raman/python3.12/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Executing run.


Running training script...


Translation task: de_core_news_sm -> en
Translation task: de_core_news_sm -> en
Translation task: de_core_news_sm -> en
Translation task: de_core_news_sm -> en
Translation task: de_core_news_sm -> en
Translation task: de_core_news_sm -> en
Translation task: de_core_news_sm -> en
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
2026-07-01 16:19:52,084 - dhcore.digitalhub.runtimes._base - ERROR - Something got wrong during function execution.
Traceback (most recent call last):
  File "/home/raman/python3.12/lib/python3.12/site-packages/digitalhub/runtimes/_base.py", line 116, in _execute
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/raman/digitalhub-tutorials/torch-translation-tutorial/runtime_python/torch-translation-tutorial/wrapper.py", line 55, in train
    raise e
  File "/home/raman/digitalhub-tutorials/torch-translation-tutorial/runtime_python/torch-translation-tut

Error running training script: 
Traceback (most recent call last):
  File "/home/raman/digitalhub-tutorials/torch-translation-tutorial/runtime_python/torch-translation-tutorial/wrapper.py", line 51, in train
    metrics = main(opts)
              ^^^^^^^^^^
  File "/home/raman/digitalhub-tutorials/torch-translation-tutorial/main.py", line 201, in main
    train_dl, valid_dl, src_vocab, tgt_vocab, _, _, special_symbols = get_data(opts)
                                                                      ^^^^^^^^^^^^^^
  File "/home/raman/digitalhub-tutorials/torch-translation-tutorial/src/data.py", line 50, in get_data
    train_iterator = _read_local_tarfile(train_file, src_lang, tgt_lang)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/raman/digitalhub-tutorials/torch-translation-tutorial/src/data.py", line 15, in _read_local_tarfile
    src_member = next(m for m in members if m.name.endswith(f".{src_lang}"))
                 ^^^^^^^^^^^^^^^^^^^

RuntimeError: Something got wrong during function execution.

In [24]:
func.run(action="build")

{'kind': 'python+build:run', 'metadata': {'project': 'translation-tutorial', 'name': '054331aef984489e97baf11cf0f1b240', 'version': 'friendly-eagle', 'created': '2026-07-01T16:12:51.66Z', 'updated': '2026-07-01T16:12:51.699Z', 'created_by': 'raman@fbk.eu', 'updated_by': 'raman@fbk.eu', 'relationships': [{'type': 'run_of', 'dest': 'store://translation-tutorial/function/python/train-translator:d0655b08afb541e88926058cbc1bda68'}]}, 'spec': {'task': 'python+build://translation-tutorial/bb2464132c564acebe5016d12b5518be', 'function': 'python://translation-tutorial/train-translator:d0655b08afb541e88926058cbc1bda68', 'source': {'source': 'git+https://github.com/scc-digitalhub/digitalhub-tutorials', 'handler': 'torch-translation-tutorial.wrapper:train', 'lang': 'python'}, 'python_version': 'PYTHON3_12', 'requirements': ['torch==2.3.0', 'torchtext==0.18.0', 'torchdata==0.9.0', 'spacy==3.8.14', 'portalocker==3.2.0', 'click >= 8.2.1'], 'inputs': {}, 'parameters': {}, 'init_parameters': {}}, 'statu

In [26]:
run = func.run(
    action="job",
    inputs={
        "training_data": project.get_artifact("train-data").key,
        "validation_data": project.get_artifact("validation-data").key
    },
    parameters={
        "epochs": 1,
        "backend": "gpu",
        "src_lang": "de_core_news_sm"
    },
    profile="1xV100",
    local_execution=False
)